In [1]:
import os
import sys
import sqlite3
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

print("DAY 12 - FINAL DEVELOPMENT")
print("=" * 55)

PROJECT_ROOT = Path("..").resolve()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
DATASET_DIR = PROJECT_ROOT / "dataset"

APP_PATH = PROJECT_ROOT / "app.py"
MODEL_PATH = ARTIFACT_DIR / "day07_optimized_pipeline.pkl"
DATASET_PATH = DATASET_DIR / "student-mat.csv"

print("Python Version:", sys.version)
print("Project Root:", PROJECT_ROOT)
print("Application Available:", APP_PATH.exists())
print("Optimized Model Available:", MODEL_PATH.exists())
print("Dataset Available:", DATASET_PATH.exists())
print("SQLite Version:", sqlite3.sqlite_version)

DAY 12 - FINAL DEVELOPMENT
Python Version: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Project Root: D:\AI GEN TECH - Intern\Student-Performance-Prediction
Application Available: True
Optimized Model Available: True
Dataset Available: True
SQLite Version: 3.53.2


In [2]:
# Inspect existing Streamlit application

app_content = APP_PATH.read_text(encoding="utf-8")

print("APPLICATION INSPECTION")
print("=" * 55)
print("File size:", APP_PATH.stat().st_size, "bytes")
print("Total lines:", len(app_content.splitlines()))

# Check important application components
components = {
    "Model loading": ["joblib.load"],
    "Session state": ["st.session_state"],
    "Prediction history": ["prediction_history"],
    "Prediction generation": [".predict("],
    "History display": ["Prediction History"],
    "CSV export": ["download_button"],
    "Clear history": ["clear"],
    "Timestamp handling": ["datetime"]
}

print("\nCOMPONENT VERIFICATION")
print("-" * 55)

for component, keywords in components.items():
    found = any(keyword in app_content for keyword in keywords)
    print(f"{component}: {'Found' if found else 'Not Found'}")

# Display relevant history-related lines
print("\nHISTORY-RELATED CODE LOCATIONS")
print("-" * 55)

for line_number, line in enumerate(app_content.splitlines(), start=1):
    if any(word in line.lower() for word in [
        "session_state", "prediction_history",
        "download_button", "clear history"
    ]):
        print(f"Line {line_number}: {line.strip()}")

APPLICATION INSPECTION
File size: 10477 bytes
Total lines: 338

COMPONENT VERIFICATION
-------------------------------------------------------
Model loading: Found
Session state: Found
Prediction history: Found
Prediction generation: Found
History display: Found
CSV export: Found
Clear history: Not Found
Timestamp handling: Found

HISTORY-RELATED CODE LOCATIONS
-------------------------------------------------------
Line 41: if "prediction_history" not in st.session_state:
Line 42: st.session_state.prediction_history = []
Line 248: st.session_state.prediction_history.append(history_record)
Line 288: history = st.session_state.prediction_history
Line 311: st.download_button(
Line 314: file_name="student_prediction_history.csv",
Line 320: st.session_state.prediction_history = []


In [3]:
import shutil

# Create a backup of the current application
backup_path = PROJECT_ROOT / "app_day11_backup.py"

shutil.copy2(APP_PATH, backup_path)

print("APPLICATION BACKUP")
print("=" * 50)
print("Backup created:", backup_path.exists())
print("Backup size:", backup_path.stat().st_size, "bytes")

# Display history-related application code
app_lines = app_content.splitlines()

print("\nPREDICTION HISTORY IMPLEMENTATION")
print("=" * 50)

for start, end in [(230, 260), (275, 338)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start - 1, min(end, len(app_lines))):
        print(f"{i + 1}: {app_lines[i]}")

APPLICATION BACKUP
Backup created: True
Backup size: 10477 bytes

PREDICTION HISTORY IMPLEMENTATION

--- Lines 230-260 ---
230:             "freetime": freetime,
231:             "goout": goout,
232:             "Dalc": Dalc,
233:             "Walc": Walc,
234:             "health": health
235:         }
236: 
237:         input_df = pd.DataFrame([input_data])
238: 
239:         try:
240:             prediction = float(model.predict(input_df)[0])
241: 
242:             # Save this successful prediction in the current session.
243:             history_record = input_data.copy()
244:             history_record["Predicted_G3"] = round(prediction, 2)
245:             history_record["Prediction_Time"] = datetime.now().strftime(
246:                 "%Y-%m-%d %H:%M:%S"
247:             )
248:             st.session_state.prediction_history.append(history_record)
249: 
250:             st.success("Prediction generated successfully!")
251: 
252:             st.markdown("### 📊 Prediction Result

In [4]:
print("APPLICATION INITIALIZATION")
print("=" * 55)

for i, line in enumerate(app_lines[:75], start=1):
    print(f"{i}: {line}")

APPLICATION INITIALIZATION
1: import os
2: from datetime import datetime
3: import joblib
4: import pandas as pd
5: import streamlit as st
6: 
7: # ---------------- PAGE CONFIGURATION ----------------
8: st.set_page_config(
9:     page_title="Student Performance Predictor",
10:     page_icon="🎓",
11:     layout="wide"
12: )
13: 
14: PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
15: MODEL_PATH = os.path.join(
16:     PROJECT_DIR, "artifacts", "day07_optimized_pipeline.pkl"
17: )
18: 
19: @st.cache_resource
20: def load_model():
21:     return joblib.load(MODEL_PATH)
22: 
23: try:
24:     model = load_model()
25: except Exception as e:
26:     st.error(f"Unable to load the optimized model: {e}")
27:     st.stop()
28: 
29: # ---------------- HEADER ----------------
30: st.title("🎓 Student Performance Prediction System")
31: st.markdown(
32:     "### Beginning-of-Term Academic Performance Prediction"
33: )
34: st.caption(
35:     "Predicting final mathematics grades using studen

In [5]:
import re

updated_app = app_content

# 1. Add SQLite and JSON imports
updated_app = updated_app.replace(
    "import os\nfrom datetime import datetime\n",
    "import os\nimport json\nimport sqlite3\nfrom datetime import datetime\n",
    1
)

# 2. Add database path and helper functions after PROJECT_DIR
database_code = '''

DB_PATH = os.path.join(
    PROJECT_DIR, "artifacts", "student_predictions.db"
)

def initialize_database():
    """Create the prediction history table if it does not exist."""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS predictions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                prediction_time TEXT NOT NULL,
                predicted_g3 REAL NOT NULL,
                student_data TEXT NOT NULL
            )
        """)

def save_prediction(input_data, prediction):
    """Save one prediction and its input values permanently."""
    prediction_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with sqlite3.connect(DB_PATH) as connection:
        connection.execute(
            """
            INSERT INTO predictions
            (prediction_time, predicted_g3, student_data)
            VALUES (?, ?, ?)
            """,
            (
                prediction_time,
                round(float(prediction), 2),
                json.dumps(input_data)
            )
        )

    return prediction_time

def get_prediction_history():
    """Retrieve saved predictions as a DataFrame."""
    with sqlite3.connect(DB_PATH) as connection:
        rows = connection.execute(
            """
            SELECT prediction_time, predicted_g3, student_data
            FROM predictions
            ORDER BY id DESC
            """
        ).fetchall()

    records = []

    for prediction_time, predicted_g3, student_data in rows:
        record = json.loads(student_data)
        record["Prediction_Time"] = prediction_time
        record["Predicted_G3"] = predicted_g3
        records.append(record)

    return pd.DataFrame(records)

def clear_prediction_history():
    """Delete all saved prediction records."""
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("DELETE FROM predictions")


initialize_database()
'''

project_dir_line = 'PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))'

assert updated_app.count(project_dir_line) == 1
updated_app = updated_app.replace(
    project_dir_line,
    project_dir_line + database_code,
    1
)

# 3. Remove temporary session-based history initialization
old_session_block = '''# ---------------- SESSION PREDICTION HISTORY ----------------
if "prediction_history" not in st.session_state:
    st.session_state.prediction_history = []
'''

assert updated_app.count(old_session_block) == 1
updated_app = updated_app.replace(old_session_block, "", 1)

# 4. Replace session-based prediction recording
old_recording = '''            # Save this successful prediction in the current session.
            history_record = input_data.copy()
            history_record["Predicted_G3"] = round(prediction, 2)
            history_record["Prediction_Time"] = datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
            st.session_state.prediction_history.append(history_record)
'''

new_recording = '''            # Save this successful prediction permanently in SQLite.
            save_prediction(input_data, prediction)
'''

assert updated_app.count(old_recording) == 1
updated_app = updated_app.replace(old_recording, new_recording, 1)

# 5. Replace history retrieval
old_history = '    history = st.session_state.prediction_history\n'
new_history = '    history_df = get_prediction_history()\n\n    history = not history_df.empty\n'

assert updated_app.count(old_history) == 1
updated_app = updated_app.replace(old_history, new_history, 1)

# Remove the old DataFrame construction because history_df is already loaded
old_dataframe_line = '        history_df = pd.DataFrame(history)\n'
assert updated_app.count(old_dataframe_line) == 1
updated_app = updated_app.replace(old_dataframe_line, "", 1)

# 6. Replace session-only clear operation with database deletion
old_clear = '''        if st.button("🗑️ Clear Prediction History"):
            st.session_state.prediction_history = []
            st.rerun()
'''

new_clear = '''        if st.button("🗑️ Clear Prediction History"):
            clear_prediction_history()
            st.success("Prediction history cleared successfully.")
            st.rerun()
'''

assert updated_app.count(old_clear) == 1
updated_app = updated_app.replace(old_clear, new_clear, 1)

# 7. Validate the prepared source without writing it
compile(updated_app, str(APP_PATH), "exec")

checks = {
    "SQLite import": "import sqlite3" in updated_app,
    "JSON import": "import json" in updated_app,
    "Database initialization": "initialize_database()" in updated_app,
    "Persistent prediction saving": "save_prediction(input_data, prediction)" in updated_app,
    "Database history retrieval": "get_prediction_history()" in updated_app,
    "Database clear operation": "clear_prediction_history()" in updated_app,
    "CSV export preserved": "st.download_button(" in updated_app,
    "Old session history removed": "st.session_state.prediction_history" not in updated_app,
    "Four input sections preserved": all(
        section in updated_app for section in [
            "Student Information",
            "Family Background",
            "Academic Information",
            "Lifestyle and Well-being"
        ]
    )
}

print("SQLITE INTEGRATION PREPARATION")
print("=" * 55)

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nOriginal file size:", len(app_content.encode("utf-8")), "bytes")
print("Prepared file size:", len(updated_app.encode("utf-8")), "bytes")
print("Python syntax: VALID" if all(checks.values()) else "Review required")

SQLITE INTEGRATION PREPARATION
SQLite import: PASS
JSON import: PASS
Database initialization: PASS
Persistent prediction saving: PASS
Database history retrieval: PASS
Database clear operation: PASS
CSV export preserved: PASS
Old session history removed: PASS
Four input sections preserved: PASS

Original file size: 10477 bytes
Prepared file size: 12142 bytes
Python syntax: VALID


In [6]:
# Save the prepared application
APP_PATH.write_text(updated_app, encoding="utf-8")

# Verify the saved file
saved_content = APP_PATH.read_text(encoding="utf-8")

assert saved_content == updated_app
compile(saved_content, str(APP_PATH), "exec")

# Check the actual database location
DB_PATH = ARTIFACT_DIR / "student_predictions.db"

# Initialize and inspect the SQLite database
with sqlite3.connect(DB_PATH) as connection:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            prediction_time TEXT NOT NULL,
            predicted_g3 REAL NOT NULL,
            student_data TEXT NOT NULL
        )
    """)

    table_info = connection.execute(
        "PRAGMA table_info(predictions)"
    ).fetchall()

print("APPLICATION SAVE VERIFICATION")
print("=" * 55)
print("Application saved:", APP_PATH.exists())
print("Application size:", APP_PATH.stat().st_size, "bytes")
print("Python syntax: VALID")
print("Backup preserved:", backup_path.exists())

print("\nDATABASE VERIFICATION")
print("=" * 55)
print("Database created:", DB_PATH.exists())
print("Database location:", DB_PATH)
print("Table columns:")

for column in table_info:
    print("-", column[1], "|", column[2])

APPLICATION SAVE VERIFICATION
Application saved: True
Application size: 12544 bytes
Python syntax: VALID
Backup preserved: True

DATABASE VERIFICATION
Database created: True
Database location: D:\AI GEN TECH - Intern\Student-Performance-Prediction\artifacts\student_predictions.db
Table columns:
- id | INTEGER
- prediction_time | TEXT
- predicted_g3 | REAL
- student_data | TEXT


In [8]:
from datetime import datetime
import json
import sqlite3
import tempfile
from pathlib import Path

# Create a separate temporary database
test_db_path = Path(tempfile.gettempdir()) / "student_prediction_test.db"

if test_db_path.exists():
    test_db_path.unlink()

sample_student = {
    "school": "GP",
    "sex": "F",
    "age": 17,
    "address": "U",
    "studytime": 2
}

sample_prediction = 12.45
sample_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 1. Create table and insert a record
with sqlite3.connect(test_db_path) as connection:
    connection.execute("""
        CREATE TABLE predictions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            prediction_time TEXT NOT NULL,
            predicted_g3 REAL NOT NULL,
            student_data TEXT NOT NULL
        )
    """)

    connection.execute(
        """
        INSERT INTO predictions
        (prediction_time, predicted_g3, student_data)
        VALUES (?, ?, ?)
        """,
        (sample_timestamp, sample_prediction, json.dumps(sample_student))
    )

# 2. Retrieve and verify
with sqlite3.connect(test_db_path) as connection:
    row = connection.execute("""
        SELECT prediction_time, predicted_g3, student_data
        FROM predictions
    """).fetchone()

retrieved_data = json.loads(row[2])

print("DATABASE INSERT AND RETRIEVAL")
print("=" * 45)
print("Record retrieved:", row is not None)
print("Timestamp:", row[0])
print("Predicted grade:", row[1])
print("Student data matches:", retrieved_data == sample_student)

# 3. Test deletion
with sqlite3.connect(test_db_path) as connection:
    connection.execute("DELETE FROM predictions")
    remaining_records = connection.execute(
        "SELECT COUNT(*) FROM predictions"
    ).fetchone()[0]

print("\nDATABASE DELETE TEST")
print("=" * 45)
print("Records remaining:", remaining_records)

# 4. Clean up
test_db_path.unlink(missing_ok=True)
print("Temporary database cleaned:", not test_db_path.exists())

DATABASE INSERT AND RETRIEVAL
Record retrieved: True
Timestamp: 2026-11-25 14:14:09
Predicted grade: 12.45
Student data matches: True

DATABASE DELETE TEST
Records remaining: 0


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\vikra\\AppData\\Local\\Temp\\student_prediction_test.db'

In [9]:
import gc
import time

gc.collect()
time.sleep(1)

try:
    test_db_path.unlink(missing_ok=True)
    print("Temporary database cleaned:", not test_db_path.exists())
except PermissionError:
    print("Temporary database is still locked.")
    print("It is safe to leave this temporary test file; it does not affect the project.")

Temporary database is still locked.
It is safe to leave this temporary test file; it does not affect the project.


In [10]:
# Inspect root-level project files and folders

print("PROJECT ROOT CONTENTS")
print("=" * 55)

for item in sorted(PROJECT_ROOT.iterdir()):
    kind = "Folder" if item.is_dir() else "File"
    print(f"{kind:8} | {item.name}")

README_PATH = PROJECT_ROOT / "README.md"
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

print("\nDOCUMENTATION STATUS")
print("=" * 55)
print("README exists:", README_PATH.exists())
print(".gitignore exists:", GITIGNORE_PATH.exists())

if README_PATH.exists():
    print("\nExisting README preview:")
    print(README_PATH.read_text(encoding="utf-8")[:1500])

if GITIGNORE_PATH.exists():
    print("\nExisting .gitignore:")
    print(GITIGNORE_PATH.read_text(encoding="utf-8"))

PROJECT ROOT CONTENTS
Folder   | .ipynb_checkpoints
File     | app.py
File     | app_day11_backup.py
Folder   | artifacts
Folder   | dataset
Folder   | models
Folder   | notebooks
Folder   | outputs
Folder   | src
File     | student.zip

DOCUMENTATION STATUS
README exists: False
.gitignore exists: False
